# Amazon Bedrock AgentCore Runtime에서 Strands 에이전트와 Instana Observability 사용하기

## 개요

이 노트북에서는 Instana 관측성을 연동한 Strands 에이전트를 Amazon Bedrock AgentCore Runtime에 배포하는 방법을 살펴봅니다. Amazon Bedrock 모델을 사용하며, OpenTelemetry(OTEL)를 통해 텔레메트리 데이터를 Instana로 전송합니다.

## 주요 구성 요소

- **Strands Agents**: 기본 텔레메트리 지원이 포함된 LLM 기반 에이전트 구축용 Python 프레임워크
- **Amazon Bedrock AgentCore Runtime**: AWS에서 에이전트를 호스팅하고 확장하기 위한 관리형 런타임 서비스
- **Instana**: OTEL을 통해 트레이스를 수신하여 애플리케이션의 실시간 관측성 및 성능을 모니터링하는 플랫폼
- **OpenTelemetry**: 텔레메트리 데이터를 수집하고 내보내기 위한 업계 표준 프로토콜

## 아키텍처

에이전트는 컨테이너로 패키징되어 호출용 HTTP 엔드포인트를 제공하는 AgentCore Runtime에 배포됩니다. 텔레메트리 데이터는 Strands 에이전트에서 OTEL exporter를 거쳐 Instana로 전달되어 모니터링과 디버깅에 사용됩니다. 이 구현에서는 Instana를 사용하기 위해 AgentCore의 기본 관측성을 비활성화합니다.

## 사전 요구 사항

- Python 3.10+
- [Amazon Bedrock AgentCore 시작하기](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-get-started-toolkit.html#agentcore-get-started-prerequisites)
- Bedrock 및 [AgentCore 권한](https://docs.aws.amazon.com/aws-managed-policy/latest/reference/BedrockAgentCoreFullAccess.html)이 구성된 AWS 자격 증명
- [Instana](https://www.ibm.com/products/instana) 계정
- 로컬에 설치된 Docker
- Amazon Bedrock 모델에 대한 액세스 권한

### 올바른 Instana Endpoint 찾기

Instana 인스턴스에 맞는 OTLP 엔드포인트를 찾으려면 다음 단계를 따릅니다.

1. Instana에서 **sidebar**를 엽니다.
2. 맨 아래로 스크롤하여 **About Instana**를 선택합니다.
3. 표시된 **Instance Region**을 확인합니다.
4. 이 리전을 기준으로 엔드포인트 [문서](https://www.ibm.com/docs/en/instana-observability/1.0.309?topic=instana-backend)에서 적절한 **HTTP OTLP endpoint(4318)**를 선택합니다.


### Instana Key 가져오기

Instana key를 가져오려면 다음 단계를 따릅니다.

1. Sidebar에서 **Agents & Collectors**를 클릭합니다.
2. **Linux – Automatic Installation (One-liner)**을 선택합니다.
3. `-a` flag 뒤에 표시되는 key를 복사합니다. 이 **Agent Key**가 **Instana Key**로 사용됩니다.

자세한 방법은 아래 이미지를 참고하세요.

<div style="text-align:left">
    <img src="../images/instana_agent_key.png" width="99%"/>
</div>

### 프로젝트 루트 디렉터리에 `.env` 파일을 생성하고 다음 환경 변수를 추가합니다.

아래 셀을 실행하면 프로젝트 루트에 `.env` 파일이 자동으로 생성됩니다. 파일이 생성되면 placeholder 값을 실제 Instana OTLP 엔드포인트와 API 키로 바꿉니다.

`.env` 파일은 API 키 및 엔드포인트와 같은 구성 정보를 안전하게 저장하므로, 값을 코드에 직접 넣지 않고 애플리케이션에서 손쉽게 불러올 수 있습니다.

**참고**: `.env` 파일을 GitHub에 commit하거나 Instana key를 공개적으로 공유하지 마세요. 자격 증명을 안전하게 보호하려면 `.gitignore` 파일에 `.env`를 추가하세요.

### .env 파일 내용 예시

In [ ]:
%%writefile .env

# Instana OTLP 엔드포인트(예: https://otlp-blue-saas.instana.io:4318)
OTEL_EXPORTER_OTLP_ENDPOINT="<instana_endpoint>"

# 참고: 이 예제는 내보내기에 StrandsTelemetry()를 사용하며 HTTP OTLP 엔드포인트만 지원합니다.
# Strands Telemetry를 사용할 때는 반드시 HTTP 엔드포인트를 사용하세요.

# Instana 키
INSTANA_KEY="<agent_key>"

아래 셀을 실행하여 종속성을 설치합니다.

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## AWS Credentials 구성

노트북을 실행하기 전에 `사전 요구 사항` 섹션의 링크를 참고하여 AWS 자격 증명을 구성하세요.

## Agent 구현

에이전트 파일(`strands_nova.py`)은 웹 검색 기능을 갖춘 여행 에이전트를 구현합니다. 주요 구성은 다음과 같습니다.
- Strands 텔레메트리 초기화

In [ ]:
%%writefile strands_nova.py
import os
import logging
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from ddgs import DDGS

logging.basicConfig(level=logging.ERROR, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(os.getenv("AGENT_RUNTIME_LOG_LEVEL", "INFO").upper())


@tool
def web_search(query: str) -> str:
    """
    Search the web for information using DuckDuckGo.

    Args:
        query: The search query

    Returns:
        A string containing the search results
    """
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=5)

        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n"
                f"   {result.get('body', 'No summary')}\n"
                f"   Source: {result.get('href', 'No URL')}\n"
            )

        return "\n".join(formatted_results) if formatted_results else "No results found."

    except Exception as e:
        return f"Error searching the web: {str(e)}"

# Bedrock 모델 초기화 함수
def get_bedrock_model():
    region = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
    model_id = os.getenv("BEDROCK_MODEL_ID", "amazon.nova-lite-v1:0")

    bedrock_model = BedrockModel(
        model_id=model_id,
        region_name=region,
        temperature=0.0,
        max_tokens=1024
    )
    return bedrock_model

# Bedrock 모델 초기화
bedrock_model = get_bedrock_model()

# 에이전트의 system prompt 정의
system_prompt = """You are an experienced travel agent specializing in personalized travel recommendations 
with access to real-time web information. Your role is to find dream destinations matching user preferences 
using web search for current information. You should provide comprehensive recommendations with current 
information, brief descriptions, and practical travel details."""

app = BedrockAgentCoreApp()

def initialize_agent():
    """올바른 텔레메트리 구성으로 에이전트를 초기화합니다."""

    # 3P 구성으로 Strands 텔레메트리 초기화
    strands_telemetry = StrandsTelemetry()
    strands_telemetry.setup_otlp_exporter()
    
    # 에이전트 생성 및 캐시
    agent = Agent(
        model=bedrock_model,
        system_prompt=system_prompt,
        tools=[web_search]
    )
    
    return agent

@app.entrypoint
def strands_agent_bedrock(payload, context=None):
    """
    페이로드로 에이전트를 호출합니다.
    """
    user_input = payload.get("prompt")
    logger.info("[%s] User input: %s", context.session_id, user_input)
    
    # 올바른 구성으로 에이전트 초기화
    agent = initialize_agent()
    
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

### AgentCore Runtime 배포 구성

이제 starter toolkit을 사용하여 AgentCore Runtime 배포를 구성합니다.
이 단계에서는 진입점, 앞서 생성한 실행 역할, requirements 파일을 설정합니다.
또한 시작할 때 Amazon ECR 리포지토리를 자동으로 생성하도록 starter toolkit을 구성합니다.

구성 명령을 실행하면 애플리케이션 코드를 기반으로 Dockerfile이 자동 생성됩니다.

**참고:**
bedrock_agentcore_starter_toolkit은 기본적으로 AgentCore Observability를 활성화합니다.
Instana를 관측성에 사용하려면 다음 섹션의 설명과 같이 기본 AgentCore Observability 구성을 제거해야 합니다.

<div style="text-align:left">
    <img src="../images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_instana_observability"
response = agentcore_runtime.configure(
    entrypoint="strands_nova.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    disable_otel=True,
)
response

### 대체 구성(미리 생성한 IAM Role 사용)

AWS 계정에서 **역할 자동 생성을 허용하지 않거나** **권한을 더 세밀하게 제어**하려는 경우, toolkit이 역할을 자동 생성하도록 하는 대신 **미리 생성한 IAM role**을 사용할 수 있습니다.

이 경우 다음과 같이 설정합니다.
- `auto_create_execution_role=False` 설정
- `execution_role` parameter에 기존 IAM role ARN 지정

이 접근 방식은 다음과 같은 경우에 유용합니다.
- 조직에서 **최소 권한 IAM policy**를 적용하는 경우
- 여러 AgentCore 배포에서 **공통 실행 역할을 재사용**하려는 경우
- 새 role 생성이 허용되지 않는 **제한된 환경**(예: 기업 또는 공유 AWS 계정)에서 작업하는 경우

수동으로 관리하는 IAM role로 런타임을 구성하는 방법은 아래 예제를 참고하세요.

In [ ]:
# 대체 구성(선택 사항)

from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

ROLE_ARN = "your_IAM_role"

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_instana_observability"
response = agentcore_runtime.configure(
    entrypoint="strands_nova.py",
    auto_create_execution_role=False,  # Auto-create role을 비활성화함
    execution_role=ROLE_ARN,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_instana_observability",
    disable_otel=True,
)
response

## AgentCore Runtime에 배포

Dockerfile이 생성되었으므로 이제 에이전트를 AgentCore Runtime에 배포합니다.

이 단계에서는 다음 작업이 자동으로 수행됩니다.
- Amazon ECR 리포지토리가 없는 경우 생성
- 컨테이너로 패키징된 에이전트를 AgentCore Runtime 환경에 배포
- 앞서 생성한 `.env` 파일에서 Instana 엔드포인트 및 API 키와 같은 환경 변수 로드

**참고:** 이 셀을 실행하기 전에 `.env` 파일이 올바르게 구성되었는지 확인하세요. 그렇지 않으면 텔레메트리 데이터가 Instana로 전송되지 않습니다.

<div style="text-align:left">
    <img src="../images/launch.png" width="75%"/>
</div>

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일에서 환경 변수 로드
load_dotenv()

# 구성 가져오기
otel_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT")
instana_key = os.getenv("INSTANA_KEY")

# Instana header 형식 지정
otel_auth_header = f"x-instana-key={instana_key}"

# AgentCore Runtime 시작
launch_result = agentcore_runtime.launch(
    env_vars={
        "BEDROCK_MODEL_ID": "amazon.nova-lite-v1:0",
        "OTEL_EXPORTER_OTLP_ENDPOINT": otel_endpoint,
        "OTEL_EXPORTER_OTLP_HEADERS": otel_auth_header,
        "OTEL_SERVICE_NAME": "AWS-APP",
        "OTEL_EXPORTER_OTLP_INSECURE": "false",
        "DISABLE_ADOT_OBSERVABILITY": "true",
    }
)

launch_result

## 배포 상태 확인

배포를 시작한 후 AgentCore Runtime 설정이 완료되기까지 몇 분 정도 걸릴 수 있습니다.
다음 코드를 사용하여 배포 상태를 실시간으로 모니터링할 수 있습니다.

이 코드는 AgentCore 엔드포인트가 다음과 같은 최종 상태에 도달할 때까지 몇 초마다 상태를 계속 확인합니다.
- READY → 배포가 성공적으로 완료됨
- CREATE_FAILED, UPDATE_FAILED 또는 DELETE_FAILED → 배포 중 오류가 발생함

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

### AgentCore Runtime 호출

마지막으로 sample prompt로 배포된 에이전트를 호출하여 응답을 테스트하고 예상대로 작동하는지 확인합니다.

<div style="text-align:left">
    <img src="../images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke(
    {"prompt": "I'm planning a weekend trip to london. What are the must-visit places and local food I should try?"}
)

아래 코드를 사용하여 에이전트의 출력을 보기 좋게 표시합니다.

In [ ]:
from IPython.display import Markdown, display

display(Markdown("".join(invoke_response["response"])))

## Instana에서 트레이스 확인

트레이스를 확인하려면 다음 단계를 따릅니다.
1. Instana 대시보드로 이동합니다.
2. Sidebar에서 `Analytics`를 클릭합니다.
3. `Hidden calls` tab 아래의 `Show internal calls` checkbox를 선택합니다.
4. `Add Filter`를 클릭하고 `Service Name`을 선택합니다.
5. 검색 창에서 "strands-agents"를 검색합니다.
6. 관련 호출 목록을 열어 분석합니다. 각 호출은 한 번의 사용자 상호 작용을 나타냅니다.
7. 호출을 클릭하여 전체 트레이스 데이터를 확인합니다.

트레이스에는 다음 정보가 포함됩니다.
- 에이전트 호출 세부 정보
- 도구 호출(웹 검색)
- 지연 시간 및 토큰 사용량을 포함한 모델 상호 작용
- 요청/응답 페이로드

## 리소스 정리(선택 사항)

테스트가 끝나면 불필요한 리소스 사용과 비용을 방지하기 위해 아래 코드로 AgentCore Runtime과 연결된 Amazon ECR 리포지토리를 삭제합니다.

In [ ]:
import boto3

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

ecr_client = boto3.client("ecr", region_name=region)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)

## 요약

Instana 관측성이 적용된 Strands 에이전트를 Amazon Bedrock AgentCore Runtime에 성공적으로 배포했습니다. 이 구현에서는 다음 내용을 살펴보았습니다.
- Strands 에이전트와 AgentCore Runtime 연동
- Instana로 트레이스를 전송하기 위한 OpenTelemetry 구성
- 텔레메트리 구성을 보장하는 올바른 초기화 순서

이제 에이전트는 Instana를 통해 완전한 관측성을 제공하는 관리형 확장 환경에서 실행됩니다.